На базе баскетбольных матчей добейтесь средней абсолютной ошибки 17 и менее очков.

In [ ]:
import re
from collections import Counter

import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

%matplotlib inline


In [ ]:
DATA_URL = 'https://storage.yandexcloud.net/aiueducation/Content/base/l10/basketball.csv'
DATA_FILE = 'basketball.csv'

gdown.download(DATA_URL, DATA_FILE, quiet=True)

basket_df = pd.read_csv(
    DATA_FILE,
    encoding='cp1251',
    sep=';',
    header=0,
    index_col=0
)

basket_df.head()


In [ ]:
match_notes = basket_df['info'].astype(str).values

print('Количество текстовых записей:', len(match_notes))


In [ ]:
MAX_WORDS = 5000


def split_text(text):
    text = text.lower()
    text = re.sub(r'[^а-яёa-z0-9\s]', ' ', text)
    return text.split()


def fit_vocab(texts, limit=MAX_WORDS):
    words = Counter()
    for item in texts:
        words.update(split_text(item))
    vocab = {'unknown': 1}
    for index, (word, _) in enumerate(words.most_common(limit - 2), start=2):
        vocab[word] = index
    return vocab


def make_bow_matrix(texts, vocab, width=MAX_WORDS):
    matrix = np.zeros((len(texts), width), dtype=np.float32)
    for row_index, item in enumerate(texts):
        for word in split_text(item):
            col_index = vocab.get(word, 1)
            if col_index < width:
                matrix[row_index, col_index] = 1.0
    return matrix


vocab = fit_vocab(match_notes)
xBOW_text = make_bow_matrix(match_notes, vocab)


In [ ]:
NUMERIC_COLUMNS = ['Ком. 1', 'Ком. 2', 'Минута', 'Секунда', 'ftime']
TARGET_COLUMN = 'fcount'

xTrain = basket_df[NUMERIC_COLUMNS].astype('int').to_numpy(dtype=np.float32)
yTrain = basket_df[TARGET_COLUMN].astype('int').to_numpy(dtype=np.float32)

print('Числовые признаки:', xTrain.shape)
print('Целевые значения:', yTrain.shape)
print('Текстовая матрица:', xBOW_text.shape)


In [ ]:
scaler = StandardScaler()
xTrain_scaled = scaler.fit_transform(xTrain).astype(np.float32)
yTrain = yTrain.astype(np.float32)

val_count = max(1, int(len(yTrain) * 0.1))
train_count = len(yTrain) - val_count

x_num_train = xTrain_scaled[:train_count]
x_text_train = xBOW_text[:train_count]
y_train = yTrain[:train_count]

x_num_val = xTrain_scaled[train_count:]
x_text_val = xBOW_text[train_count:]
y_val = yTrain[train_count:]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_dataset = TensorDataset(
    torch.tensor(x_num_train, dtype=torch.float32),
    torch.tensor(x_text_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
)

val_dataset = TensorDataset(
    torch.tensor(x_num_val, dtype=torch.float32),
    torch.tensor(x_text_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32).view(-1, 1)
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)


In [ ]:
class BasketScoreModel(nn.Module):
    def __init__(self, num_size, text_size):
        super().__init__()
        self.num_part = nn.Sequential(
            nn.Linear(num_size, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 64),
            nn.ReLU()
        )
        self.text_part = nn.Sequential(
            nn.Linear(text_size, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.result_part = nn.Sequential(
            nn.Linear(320, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x_num, x_text):
        a = self.num_part(x_num)
        b = self.text_part(x_text)
        x = torch.cat([a, b], dim=1)
        return self.result_part(x)


model_final = BasketScoreModel(
    num_size=xTrain_scaled.shape[1],
    text_size=xBOW_text.shape[1]
).to(device)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model_final.parameters(), lr=0.0005)

model_final


In [ ]:
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    total_mae = 0.0
    total_count = 0

    with torch.set_grad_enabled(is_train):
        for x_num_batch, x_text_batch, y_batch in loader:
            x_num_batch = x_num_batch.to(device)
            x_text_batch = x_text_batch.to(device)
            y_batch = y_batch.to(device)

            y_pred = model(x_num_batch, x_text_batch)
            loss = loss_fn(y_pred, y_batch)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            batch_size = y_batch.size(0)
            total_loss += loss.item() * batch_size
            total_mae += torch.abs(y_pred - y_batch).sum().item()
            total_count += batch_size

    return total_loss / total_count, total_mae / total_count


def train_model(model, train_loader, val_loader, epochs=30):
    history = {'loss': [], 'mae': [], 'val_loss': [], 'val_mae': []}

    for epoch in range(1, epochs + 1):
        train_loss, train_mae = run_epoch(model, train_loader, optimizer)
        val_loss, val_mae = run_epoch(model, val_loader)

        history['loss'].append(train_loss)
        history['mae'].append(train_mae)
        history['val_loss'].append(val_loss)
        history['val_mae'].append(val_mae)

        print(
            f'Эпоха {epoch:02d}/30 | '
            f'loss: {train_loss:.4f} | mae: {train_mae:.4f} | '
            f'val_loss: {val_loss:.4f} | val_mae: {val_mae:.4f}'
        )

    return history


history = train_model(model_final, train_loader, val_loader, epochs=30)


In [ ]:
def plot_training_mae(train_history):
    plt.figure(figsize=(10, 5))
    plt.plot(train_history['mae'], label='Ошибка на обучении')
    plt.plot(train_history['val_mae'], label='Ошибка на проверке')
    plt.xlabel('Эпоха')
    plt.ylabel('MAE, очки')
    plt.title('Изменение средней абсолютной ошибки')
    plt.legend()
    plt.grid(True)
    plt.show()


plot_training_mae(history)


In [ ]:
def predict_values(model, x_num, x_text, batch_size=256):
    dataset = TensorDataset(
        torch.tensor(x_num, dtype=torch.float32),
        torch.tensor(x_text, dtype=torch.float32)
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    preds = []

    model.eval()
    with torch.no_grad():
        for x_num_batch, x_text_batch in loader:
            x_num_batch = x_num_batch.to(device)
            x_text_batch = x_text_batch.to(device)
            batch_pred = model(x_num_batch, x_text_batch).cpu().numpy().ravel()
            preds.append(batch_pred)

    return np.concatenate(preds)


def show_mae_for_two_inputs(model, x_num, x_text, y_true, plot=False):
    y_pred = predict_values(model, x_num, x_text)
    mae = np.mean(np.abs(y_true - y_pred))
    percent = mae / y_true.mean(axis=0) * 100

    print(
        'Средняя абсолютная ошибка {:.3f} очков, это {:.3f}% от средней суммы очков по {} играм'.format(
            mae,
            percent,
            len(x_num)
        )
    )

    if plot:
        plt.figure(figsize=(7, 7))
        plt.scatter(y_true, y_pred, alpha=0.5)
        plt.xlabel('Правильные значения')
        plt.ylabel('Предсказания')
        plt.axis('equal')
        plt.plot([0, 250], [0, 250])
        plt.show()

    return mae


In [ ]:
predictions = predict_values(model_final, xTrain_scaled, xBOW_text)
final_mae = np.mean(np.abs(yTrain - predictions))

print(f'Итоговая средняя ошибка (MAE): {round(final_mae, 2)} очков')

if final_mae <= 17:
    print('Ошибка менее 17.')
else:
    print('Нужно еще немного дообучить или усложнить модель.')

show_mae_for_two_inputs(
    model_final,
    xTrain_scaled,
    xBOW_text,
    yTrain,
    plot=True
)


In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(yTrain, predictions, alpha=0.5)
plt.plot(
    [min(yTrain), max(yTrain)],
    [min(yTrain), max(yTrain)]
)
plt.xlabel('Правильные значения')
plt.ylabel('Предсказания')
plt.title('Анализ предсказаний счета матчей')
plt.show()
